- structured output / bind tools
    - langchain -> openai api -> http request
- 某种意义上的接口的一个约定（SDD，Specification Driven Dev）
  - 复杂的 agent workflow，module 之间的接口（字段）约定
- 我们基于 pydantic BaseModel 定义了我们要格式化输出的 data model，且 `llm.with_structured_output(XXModel)` 之后确实不需要再在 `system prompt` 中显式地指定 output format 了或者指定 function/tool 的信息；
- https://python.langchain.com/docs/how_to/structured_output/
    - Beyond just the structure of the Pydantic class, the following are necessary and important.
        - **the name of the Pydantic class**,
        - **the docstring**,
        - and **the names and provided descriptions of parameters**
    - Most of the time `with_structured_output` is using **a model's function/tool calling API**, and you can effectively think of all of this information as being added to the model prompt.
- misc
    - 通过格式化的 response format 实现 CoT（step by step reasoning）
        - https://platform.openai.com/docs/guides/structured-outputs#examples

- 1203 update (`include_raw`)
```python
structured_llm = llm.with_structured_output(Joke, include_raw=True)

result = structured_llm.invoke("Tell me a joke about cats")
# result is now a dict with 'raw', 'parsed', and 'parsing_error' keys
print("--- Raw Output ---")
print(result["raw"].content)
print("\n--- Parsed Output ---")
print(result["parsed"])
```

### openai api 

- https://platform.openai.com/docs/guides/structured-outputs
    - response_format
- https://platform.openai.com/docs/guides/function-calling
    - tools

In [9]:
from pydantic import BaseModel
import json

In [2]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [13]:
print(json.dumps(CalendarEvent.model_json_schema(), indent=2))

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "date": {
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


In [6]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

In [7]:
@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

In [18]:
print(json.dumps(convert_to_openai_tool(add), indent=2))

{
  "type": "function",
  "function": {
    "name": "add",
    "description": "Add two integers.",
    "parameters": {
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object"
    }
  }
}


### langchain: with_structured_output/bind_tools

> scripts/test_structure_outputs.py

- method: 'json_schema', 'function_calling', 'json_mode'
    - json_schema: 强校验；默认
    - function_calling: OpenAI 请求里会出现 `tools=[{ "type":"function", "function":{ "name": "...", "parameters": {...} } }]`
        - 把一个数据类的实例化当做函数
    - JSON mode: 最宽松，仅要求“返回合法 JSON”，不强制匹配 schema
    - 返回时，直接可以实例化出一个对应的 pydantic model
- api 请求
    - 对于 json_schema
    ```
    "response_format": {
      "type": "json_schema",
      "json_schema": {
        "name": "YourSchemaName",
        "strict": true,
        "schema": { ... 你的 JSON Schema ... }
      }
    }
    ```
    - 对于 function_calling
    ```
      "tool_choice": {
        "type": "function",
        "function": {
          "name": "Item"
        }
      },
      "tools": [
        {
          "type": "function",
          "function": {
            "name": "Item",
            "description": "",
            "parameters": {
              "properties": {
                "title": {
                  "type": "string"
                },
                "year": {
                  "type": "integer"
                }
              },
              "required": [
                "title",
                "year"
              ],
              "type": "object"
            }
          }
        }
      ]
    ```

#### error code: 400 (invalid_json_schema)

- llm 输出的结构化 json，missing 了一些 pydantic 定义的 models 的（required）字段
    - 核心就在于这个字段可能是 `Dict[str, int]`，一个可行的方案是将其转为 `List[xx]`

#### bind_tools

```python
from langgraph.prebuilt import create_react_agent
# 内部也是通过 bind_tools 的方式封装到 llm 中的
agent = create_react_agent(
    model=llm,
    tools=tools,
)
```

- 本质上再调用 api 时，会填入
    - `tools`：字段内

### pydantic basics

- `...`
    - ... 是 Python 的 Ellipsis 对象。在 Pydantic 里，把它放在 Field(...) 的第一个位置，表示“此字段必填，没有默认值”。

### pydantic model

- model_validate/model_validate_json：校验并实例化

In [6]:
from typing import List, Optional
from datetime import datetime
from pydantic import BaseModel, Field, ValidationError

class Item(BaseModel):
    id: int
    name: str
    price: float = Field(gt=0)              # 校验：必须 > 0
    tags: List[str] = []
    created_at: datetime                    # 自动把 ISO 字符串转 datetime

class Order(BaseModel):
    order_id: str
    items: List[Item]
    user_email: str = Field(alias="userEmail")  # 支持 JSON 字段别名
    note: Optional[str] = None

    # 允许用别名入参；忽略 JSON 里多余字段
    model_config = {"populate_by_name": True, "extra": "ignore"}

# -------- 从 JSON 字符串直接创建 --------
json_str = """
{
  "order_id": "A-001",
  "items": [
    {"id": 1, "name": "Pen", "price": 1.2, "tags": ["stationery"], "created_at": "2025-10-28T08:30:00Z"}
  ],
  "userEmail": "alice@example.com",
  "unknown_field": "will be ignored",
  "test_key": "test_value"
}
"""

In [7]:
order = Order.model_validate_json(json_str)
print(order)                     # Order(order_id='A-001', items=[...], user_email='alice@example.com', ...)

order_id='A-001' items=[Item(id=1, name='Pen', price=1.2, tags=['stationery'], created_at=datetime.datetime(2025, 10, 28, 8, 30, tzinfo=TzInfo(UTC)))] user_email='alice@example.com' note=None


In [9]:
import json
data = json.loads(json_str)
order2 = Order.model_validate(data)
order2

Order(order_id='A-001', items=[Item(id=1, name='Pen', price=1.2, tags=['stationery'], created_at=datetime.datetime(2025, 10, 28, 8, 30, tzinfo=TzInfo(UTC)))], user_email='alice@example.com', note=None)

In [10]:
print(order.model_dump())                         # 标准字段名
print(order.model_dump_json(indent=2, by_alias=True))  # 使用别名输出

{'order_id': 'A-001', 'items': [{'id': 1, 'name': 'Pen', 'price': 1.2, 'tags': ['stationery'], 'created_at': datetime.datetime(2025, 10, 28, 8, 30, tzinfo=TzInfo(UTC))}], 'user_email': 'alice@example.com', 'note': None}
{
  "order_id": "A-001",
  "items": [
    {
      "id": 1,
      "name": "Pen",
      "price": 1.2,
      "tags": [
        "stationery"
      ],
      "created_at": "2025-10-28T08:30:00Z"
    }
  ],
  "userEmail": "alice@example.com",
  "note": null
}
